In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, LSTM, Dense, Dropout, BatchNormalization, LeakyReLU
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, cohen_kappa_score, matthews_corrcoef

In [ ]:
df = pd.read_csv("/kaggle/input/adasyn-1/train_adasyn_only.csv")

In [ ]:
selected_columns = ['hv005', 'hv021', 'hv023', 'hv105', 'hv115', 'hv104', 'hv106', 'hml18',
                    'shb70', 'ha53', 'shb13', 'hv206', 'hv116', 'avg_sys', 'avg_dia', 'bmi']

In [ ]:
#split the data into training data testing sets
X = df[selected_columns]
y = df['final_diabetes']

In [ ]:
print("Input Shape: ", X.shape)
print("y Shape: ", y.shape)

In [ ]:
# Convert X and y to numpy arrays if they are not already
X = np.array(X)
y = np.array(y)

In [ ]:
# Standardize the data
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
# Reshape data for the CNN-LSTM hybrid model
X = X.reshape((X.shape[0], X.shape[1], 1))
X.shape

In [ ]:
# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model.save("/kaggle/working/Shap/Shap.keras")

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model(
    "/kaggle/input/models/azim069/keras/keras/default/1/Shap.keras"
)

print("Model loaded successfully.")
model.summary()

In [ ]:
import shap

In [ ]:
# Function to make predictions (needed for KernelExplainer)
def predict_proba(data):
    data_reshaped = data.reshape((data.shape[0], data.shape[1], 1))
    return model.predict(data_reshaped).flatten()

# Select a subset of the training data as the background dataset for SHAP
X_train_sample = X_train[:200].reshape(200, -1)  # Adjust the sample size if needed

# Select a subset of the test data for SHAP explanations
X_test_sample = X_test[:100].reshape(100, -1)  # Adjust the sample size if needed

print(X_train_sample.shape)
print(X_test_sample.shape)

In [ ]:
# Create a SHAP KernelExplainer
explainer = shap.KernelExplainer(predict_proba, X_train_sample)

In [ ]:
# Compute SHAP values for the test sample
shap_values = explainer.shap_values(X_test_sample)

In [ ]:
import os
import shap
import matplotlib.pyplot as plt

# Create directory if not exists
os.makedirs('/kaggle/working/Fig', exist_ok=True)

# Set bold font globally
plt.rcParams.update({
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold'
})

print("SHAP summary plot for Class 1")

# Generate SHAP plot
shap.summary_plot(shap_values, X_test_sample, feature_names=selected_columns)

# Get current figure
fig = plt.gcf()

# Save files
fig.savefig('/kaggle/working/Fig/New_shap_summary_8.pdf', bbox_inches='tight')
fig.savefig('/kaggle/working/Fig/New_shap_summary_8.svg', bbox_inches='tight')

plt.show()
plt.close(fig)


In [ ]:
import os
import shap
import matplotlib.pyplot as plt
import numpy as np

# Create folder if not exists
os.makedirs('/kaggle/working/Fig', exist_ok=True)

plt.rcParams.update({
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold'
})

print("SHAP summary plot for Class 1")

# -------------------------------
# Handle multi-class safely
# -------------------------------
if isinstance(shap_values, list):
    sv = shap_values[1]   # Class 1
else:
    sv = shap_values

# If SHAP Explanation object
if hasattr(sv, "values"):
    sv = sv.values

# -------------------------------
# Ensure row match
# -------------------------------
n = sv.shape[0]
X_plot = X_test[:n]

# -------------------------------
# Flatten 3D -> 2D for CNN/LSTM
# (N,16,1) → (N,16)
# -------------------------------
if len(X_plot.shape) == 3:
    X_plot = X_plot.reshape(X_plot.shape[0], X_plot.shape[1])
    sv = sv.reshape(sv.shape[0], sv.shape[1])

# -------------------------------
# Create SHAP plot
# -------------------------------
shap.summary_plot(sv, X_plot, feature_names=selected_columns, show=False)

# Save figure
fig = plt.gcf()
fig.savefig('/kaggle/working/Fig/New_shap_summary_8_1.pdf', bbox_inches='tight')
fig.savefig('/kaggle/working/Fig/New_shap_summary_8_1.svg', bbox_inches='tight')

plt.show()
plt.close(fig)


In [ ]:
# Create the SHAP summary plot
plt.figure()
shap.summary_plot(shap_values, X_test_sample, feature_names=selected_columns, show=False)

# Customize the plot
ax = plt.gca()

# Create custom y-tick labels to ensure correct ordering
y_ticks = ax.get_yticks()
y_labels = [f"$\\mathbf{{{label}}}$" for label in np.array(selected_columns)[::-1]]

# Apply custom y-tick labels
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels, fontweight='bold')

# Customize x-axis and y-axis labels
ax.set_ylabel("Feature value", fontweight='bold')
ax.set_xlabel("SHAP value (impact on model output)", fontweight='bold')


# Save the plot as JPG
plt.savefig('shap_summary_plot_custom.jpg', format='jpg', bbox_inches='tight')
plt.show()
plt.close()